# ST-EEGFormer Pretrained + LoRA

This notebook uses the official ST-EEGFormer repo plus the Hugging Face pretrained encoder weights, then adds LoRA adapters to attention blocks and a binary classifier head for the Congruent vs Incongruent Pure task.


In [ ]:
import os
# Set environment/thread limits BEFORE importing heavy numerical libraries to avoid
# deadlocks and 'Canceled future for execute_request' errors in constrained runtimes.
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import sys
import subprocess
import warnings
from pathlib import Path

%pip install -q matplotlib
import matplotlib.pyplot as plt
%pip install -q mne
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch

NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (
    Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
    if "google.colab" in sys.modules
    else NOTEBOOK_DIR.parent.parent / "src"
)

try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive", force_remount=True)
except ImportError:
    IN_COLAB = False

import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# Install lightweight, torch-free deps always safe in Colab
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "safetensors", "huggingface_hub", "--no-deps"], check=True)
    print("Colab detected — skipping torch/torchvision reinstall (use preinstalled build).")
else:
    # Non-Colab: install full deps (only if needed)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "timm", "safetensors", "huggingface_hub"], check=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from br_analysis import BinaryTaskConfig, RawEpochSpec, detect_project_paths, load_raw_epoch_dataset
from br_analysis.pretrained import BackboneClassifier, prepare_lora_backbone, load_backbone_weights, align_channels_to_steegformer
from br_analysis.torch_training import (
    TorchTrainConfig,
    cross_validate_torch_model,
    plot_fold_metrics,
    plot_torch_history,
)
from br_analysis.torch_models import count_trainable_parameters

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available; using CPU.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
CUDA is not available; using CPU.


## Task And Environment


In [16]:
# ST-EEGFormer needs the Hugging Face downloader and the official repo code.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "safetensors", "timm"], check=True)

TASK_TAG = "c_vs_ip_st-eegformer_lora"
paths = detect_project_paths("c_vs_i_steegformer_pretrained_lora", in_colab=IN_COLAB)
SELECTED_SUBJECTS_CSV = paths.analysis_root / "01_exploracio" / "tables" / "subjectes_seleccionats.csv"
MAX_SUBJECTS = 22

REPO_DIR = paths.analysis_dir / "external" / "STEEGFormer"
WEIGHTS_DIR = paths.cache_dir / "steegformer_weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/LiuyinYang1101/STEEGFormer", str(REPO_DIR)], check=True)

# Some repos nest their python files. Let's add REPO_DIR and immediate subdirs to sys.path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
for subdir in REPO_DIR.iterdir():
    if subdir.is_dir() and str(subdir) not in sys.path:
        sys.path.insert(0, str(subdir))

# Torchvision is a dependency of the ST-EEGFormer code
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchvision", "--upgrade"], check=True)
# import torchvision
from models_mae_eeg import mae_vit_base_patch16
from huggingface_hub import hf_hub_download

TASK = BinaryTaskConfig(
    name=TASK_TAG,
    class_0="Congruent",
    class_1="Incongruent_Pur",
    event_codes={"Congruent": [1, 2], "Incongruent_Pur": [3, 4, 5, 6]},
    selected_subjects_csv=SELECTED_SUBJECTS_CSV,
    max_subjects=MAX_SUBJECTS,
    undersample_classes=True,
    analysis_window=(0.0, 1.5),
    random_seed=RANDOM_SEED,
)

print(f"Projecte: {paths.project_dir}")
print(f"Resultats: {paths.analysis_dir}")
print(f"Repo ST-EEGFormer: {REPO_DIR}")
print(f"Weights dir: {WEIGHTS_DIR}")

Projecte: H:\.shortcut-targets-by-id\19KyEKxqClpX2n1-oU8TXRrmUspaPJKNU\TFG_Nils_2026
Resultats: H:\.shortcut-targets-by-id\19KyEKxqClpX2n1-oU8TXRrmUspaPJKNU\TFG_Nils_2026\04_Anàlisi\c_vs_i_steegformer_pretrained_lora
Repo ST-EEGFormer: H:\.shortcut-targets-by-id\19KyEKxqClpX2n1-oU8TXRrmUspaPJKNU\TFG_Nils_2026\04_Anàlisi\c_vs_i_steegformer_pretrained_lora\external\STEEGFormer
Weights dir: H:\.shortcut-targets-by-id\19KyEKxqClpX2n1-oU8TXRrmUspaPJKNU\TFG_Nils_2026\04_Anàlisi\c_vs_i_steegformer_pretrained_lora\cache\steegformer_weights


## Load Raw Epochs


In [17]:
spec = RawEpochSpec(picks=None, bandpass=(1.0, 40.0), resample_sfreq=128.0)
dataset = load_raw_epoch_dataset(paths, TASK, spec)
X_raw = np.asarray(dataset.X, dtype=np.float32)
y = np.asarray(dataset.y, dtype=np.int64)
ch_names = list(dataset.channel_names)
times = dataset.times

# Pad/truncate to the ST-EEGFormer reference length (6s @ 128Hz = 768 samples).
TARGET_LEN = 768
if X_raw.shape[-1] < TARGET_LEN:
    X_raw = np.pad(X_raw, ((0, 0), (0, 0), (0, TARGET_LEN - X_raw.shape[-1])), mode="constant")
else:
    X_raw = X_raw[:, :, :TARGET_LEN]

print(f"Raw X shape before channel alignment: {X_raw.shape}")
print(f"Classes: {np.bincount(y)}")

Subjectes seleccionats a carregar: 22
Mostres totals: 5616 | Congruent=2808 | Incongruent_Pur=2808
Subjectes carregats: 22 | Canals: 60
Raw X shape before channel alignment: (5616, 60, 768)
Classes: [2808 2808]


In [ ]:
# Align channels to the 142-channel ST-EEGFormer format.
align_channels_to_steegformer = align_channels_to_steegformer

print(f"Channel names: {ch_names}")
print(f"Before alignment: X_raw shape = {X_raw.shape}")

X = align_channels_to_steegformer(X_raw, ch_names)
assert X.shape[1] == 142, f"Expected 142 aligned channels, got {X.shape[1]}"
print(f"After alignment: X shape = {X.shape}")

Channel names: ['Fp1', 'Fp2', 'AF7', 'AF3', 'AFz', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'FT8', 'T7', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'T8', 'TP7', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'TP8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'O1', 'Oz', 'O2']
Before alignment: X_raw shape = (5616, 60, 768)
Aligned 60/60 channels to 142-channel format.
After alignment: X shape = (5616, 142, 768)


## Build Pretrained Model


In [ ]:
# Descarregar model base ST-EEGFormer
checkpoint_path = Path(
    hf_hub_download(
        repo_id="eugenehp/ST-EEGFormer",
        filename="ST-EEGFormer_base_encoder.safetensors",
        local_dir=str(WEIGHTS_DIR),
        local_dir_use_symlinks=False,
    )
)
print(f"Checkpoint: {checkpoint_path}")

c:\Users\nils\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:834: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Checkpoint: H:\.shortcut-targets-by-id\19KyEKxqClpX2n1-oU8TXRrmUspaPJKNU\TFG_Nils_2026\04_Anàlisi\c_vs_i_steegformer_pretrained_lora\cache\steegformer_weights\ST-EEGFormer_base_encoder.safetensors


: 

## Train


In [ ]:
def make_model():
    backbone = mae_vit_base_patch16()
    load_backbone_weights(backbone, checkpoint_path, strict=False)
    prepare_lora_backbone(
        backbone,
        include_substrings=("attn",),
        exclude_substrings=("head", "classifier", "decoder", "patch_embed", "mask_token"),
        r=8,
        alpha=16.0,
        dropout=0.05,
        train_classifier=False,
    )
    model = BackboneClassifier(backbone, n_classes=2, example_input=torch.from_numpy(X[:1]))
    trainable, total = count_trainable_parameters(model)
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


train_cfg = TorchTrainConfig(
    epochs=30,
    batch_size=4,
    lr=2e-4,
    weight_decay=1e-4,
    patience=6,
    n_splits=5,
    val_size=0.2,
    random_seed=RANDOM_SEED,
    device=DEVICE,
    progress=True,
)

metrics, history = cross_validate_torch_model(X, y, make_model, train_cfg)
metrics_path = paths.tables_dir / f"cv_metrics_{TASK_TAG}.csv"
history_path = paths.tables_dir / f"training_history_{TASK_TAG}.csv"
metrics.to_csv(metrics_path, index=False)
history.to_csv(history_path, index=False)
print(f"Saved metrics: {metrics_path}")
print(f"Saved history: {history_path}")
metrics

Using torch device: cpu
Trainable parameters: 443,906 / 111,321,874 (0.40%)


## Visualize Training


In [ ]:
plot_fold_metrics(metrics, output_file=paths.figures_dir / f"fold_metrics_{TASK_TAG}.png")
plot_torch_history(history, output_file=paths.figures_dir / f"training_history_{TASK_TAG}.png")

summary = metrics.drop(columns=["fold"]).agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})
summary_path = paths.tables_dir / f"summary_{TASK_TAG}.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")
summary